# Block 08: PC Interpretation and Centering

**Goal**  
Interpret leading weighted components conservatively and quantify how centering changes those directions.

**Checklist alignment**  
Uses the real K-alpha dataset to support cautious discussion-level interpretation work.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [2]:
out = run_block08_pc_suite(cfg)
display(out["overlap_df"].pivot_table(index=["fit_variant", "component"], columns="proxy", values="weighted_cosine"))
display(out["corr_df"].pivot_table(index="component", columns="proxy", values="pearson_r"))

proxy                  mean_like  template_like  timing_like  width_like
fit_variant component                                                   
centered    PC1         0.267197       0.267197     0.729057    0.608936
            PC2         0.006356       0.006356     0.054300    0.049566
            PC3         0.018764       0.018764     0.006861    0.004970
uncentered  PC1         0.999995       0.999995     0.349545    0.075140
            PC2         0.000209       0.000209     0.659099    0.611257
            PC3         0.000133       0.000133     0.060739    0.056774

proxy,amplitude_proxy,rq_A,rq_time_shift,timing_proxy,width_proxy
component,,,,,
PC1,-0.999998,-0.960237,0.000729,0.005396,0.190909
PC2,0.019159,0.012663,0.528216,-0.865394,0.218499
PC3,0.009954,0.012775,0.124850,-0.173635,-0.057189


In [3]:
overlap_pivot = out["overlap_df"].pivot_table(index=["fit_variant", "component"], columns="proxy", values="weighted_cosine")
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(overlap_pivot.to_numpy(), aspect="auto", cmap="viridis")
ax.set_xticks(range(len(overlap_pivot.columns)), overlap_pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(overlap_pivot.index)), [f"{a} / {b}" for a, b in overlap_pivot.index])
ax.set_title("PC overlap matrix")
fig.colorbar(im, ax=ax, label="weighted cosine")
save_figure(fig, dirs["figures"] / "block_08_pc_overlap_matrix.png")
plt.close(fig)

In [4]:
overlap_path = dirs["tables"] / "block_08_pc_metrics.csv"
corr_path = dirs["tables"] / "block_08_pc_correlations.csv"
manifest_path = dirs["manifests"] / "block_08_pc_summary.json"
save_dataframe(out["overlap_df"], overlap_path)
save_dataframe(out["corr_df"], corr_path)
save_json(out["summary"], manifest_path)
pd.DataFrame([manifest_row("block_08", "real-support", str(overlap_path.relative_to(REPO_ROOT)), cfg)])

,block_id,regime,output_path,seed,trace_len,pretrigger
0,block_08,real-support,results/tables/block_08_pc_metrics.csv,314159,32768,4000
